# Ex 1

### 1. Definirea Variabilelor și Ipotezelor
Definim trei variabile discrete cu valori în $\{1, 2, 3\}$:
- **$C$ (Contestant):** Alegerea concurentului, $P(C = c) = \frac{1}{3}$.
- **$P$ (Prize):** Ușa cu premiul, $P(P = p) = \frac{1}{3}$.
- **$H$ (Host):** Ușa deschisă de prezentator ($H \neq C$ și $H \neq P$).

**Regulile pentru $H \mid C, P$:**
- Dacă $P = C \implies P(H = h \mid C=c, P=c) = \frac{1}{2}$ (prezentatorul alege la întâmplare dintre celelalte 2 uși).
- Dacă $P \neq C \implies P(H = h \mid C=c, P=p) = 1$ (prezentatorul are o singură ușă disponibilă).

---

### 2. Demostrația Matematică (Bayes)
Presupunem că concurentul alege **Ușa 1** ($C = 1$), iar prezentatorul deschide **Ușa 3** ($H = 3$).

Aflăm probabilitatea ca premiul să fie la ușa $p$:
$$P(P = p \mid C = 1, H = 3) = \frac{P(H = 3 \mid C = 1, P = p) \cdot P(P = p)}{P(H = 3 \mid C = 1)}$$

> **Justificarea simplificării:** Din definiția Bayes, numărătorul conține $P(P = p, C = 1) = P(P = p) \cdot P(C = 1)$ datorită independenței apriori $P \perp C$. Termenul $P(C = 1)$ de la numărător se simplifică cu cel de la numitor, $P(C = 1, H = 3) = P(H = 3 \mid C = 1) \cdot P(C = 1)$.

1. **Marginalizarea numitorului:**
   $$P(H = 3 \mid C = 1) = \left(\frac{1}{2} \cdot \frac{1}{3}\right) + \left(1 \cdot \frac{1}{3}\right) + \left(0 \cdot \frac{1}{3}\right) = \frac{1}{2}$$

2. **Probabilitatea păstrării opțiunii ($P = 1$):**
   $$P(P = 1 \mid C = 1, H = 3) = \frac{\frac{1}{6}}{\frac{1}{2}} = \frac{1}{3}$$

3. **Probabilitatea schimbării opțiunii ($P = 2$):**
   $$P(P = 2 \mid C = 1, H = 3) = \frac{\frac{1}{3}}{\frac{1}{2}} = \frac{2}{3}$$

> **Concluzie:** Schimbarea alegerii dublează șansa de câștig ($\frac{2}{3}$ față de $\frac{1}{3}$).

---

### 3. Implementarea în `pgmpy`
- **Graful (DAG):** $C \to H \leftarrow P$ ($C$ și $P$ sunt independente apriori: $C \perp P$).
- **CPT-uri:** $C$ și $P$ au distribuții uniforme ($1/3$), iar $H$ are un tabel de probabilități condiționate de dimensiune $3 \times 9$.
- **Inferență:** `VariableElimination` fixează $C=1, H=3$ și returnează distribuția aposteriori pentru $P$: $[1/3, 2/3, 0]^T$.

In [8]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
import random
import scipy.stats as stats

model = DiscreteBayesianNetwork([
    ('B', 'T'),
    ('B', 'X'),
    ('B', 'D'),
    ('T', 'D')
])

cpd_b = TabularCPD(
    variable='B', 
    variable_card=2, 
    values=[[0.9], [0.1]]
)

cpd_t = TabularCPD(
    variable='T', 
    variable_card=2,
    values=[[0.7, 0.2],
            [0.3, 0.8]],
    evidence=['B'], 
    evidence_card=[2]
)

cpd_x = TabularCPD(
    variable='X', 
    variable_card=2,
    values=[[0.9, 0.1],
            [0.1, 0.9]],
    evidence=['B'], 
    evidence_card=[2]
)

cpd_d = TabularCPD(
    variable='D', 
    variable_card=2,
    values=[[0.9, 0.5, 0.6, 0.1],
            [0.1, 0.5, 0.4, 0.9]],
    evidence=['B', 'T'], 
    evidence_card=[2, 2]
)

model.add_cpds(cpd_b, cpd_t, cpd_x, cpd_d)
assert model.check_model()

infer = VariableElimination(model)
result_b = infer.query(variables=['B'], evidence={'T': 1, 'D': 1})
print("P(B | T=1, D=1)")
print(result_b)

result_c = infer.query(variables=['X'], evidence={'B': 0, 'T': 1, 'D': 1})
print("\nP(X | B=0, T=0, D=1)")
print(result_c)

P(B | T=1, D=1)
+------+----------+
| B    |   phi(B) |
+======+==========+
| B(0) |   0.6522 |
+------+----------+
| B(1) |   0.3478 |
+------+----------+

P(X | B=0, T=0, D=1)
+------+----------+
| X    |   phi(X) |
+======+==========+
| X(0) |   0.9000 |
+------+----------+
| X(1) |   0.1000 |
+------+----------+


In [7]:
def simulate_game():
    starter = random.choice([0, 1])
    
    n = random.randint(1, 6)
    
    other = 1 - starter
    p_stema = (4 / 7) if other == 1 else 0.5
    
    m = sum(1 for _ in range(2 * n) if random.random() < p_stema)
    
    winner = starter if n >= m else other
    return winner

wins = {0: 0, 1: 0}
N_SIM = 10000

for _ in range(N_SIM):
    wins[simulate_game()] += 1

print(f"Câștiguri J0: {wins[0]} ({wins[0] / N_SIM:.2%})")
print(f"Câștiguri J1: {wins[1]} ({wins[1] / N_SIM:.2%})")


game_model = DiscreteBayesianNetwork([
    ('S', 'M'),
    ('N', 'M')
])

cpd_s = TabularCPD(variable='S', variable_card=2, values=[[0.5], [0.5]])

cpd_n = TabularCPD(
    variable='N', 
    variable_card=6,
    values=[[1/6] for _ in range(6)]
)

m_values = []
for m in range(13):
    row = []
    for s in [0, 1]:
        p_head = (4 / 7) if s == 0 else 0.5 
        for n_idx in range(6):
            n_flips = 2 * (n_idx + 1)
            if m <= n_flips:
                p_m = stats.binom.pmf(m, n_flips, p_head)
            else:
                p_m = 0.0
            row.append(p_m)
    m_values.append(row)

cpd_m = TabularCPD(
    variable='M', 
    variable_card=13,
    values=m_values,
    evidence=['S', 'N'], 
    evidence_card=[2, 6]
)

game_model.add_cpds(cpd_s, cpd_n, cpd_m)
assert game_model.check_model()

infer_game = VariableElimination(game_model)
result_s = infer_game.query(variables=['S'], evidence={'M': 1})

print("\nProbabilitatea Starter-ului pentru M = 1")
print(result_s)

Câștiguri J0: 4275 (42.75%)
Câștiguri J1: 5725 (57.25%)

Probabilitatea Starter-ului pentru M = 1
+------+----------+
| S    |   phi(S) |
+======+==========+
| S(0) |   0.4529 |
+------+----------+
| S(1) |   0.5471 |
+------+----------+
